In [2]:
# ============================================================
# TASK 4 - NORMALIZATION ANALYSIS
# Amazon ML Challenge 2026
# Business Entity Resolution
#
# Purpose:
#   1. Test candidate normalization transformations
#   2. Measure positive-pair recovery
#   3. Measure negative-pair collision risk
#   4. Identify transformations safe enough for production
#
# IMPORTANT:
#   This notebook does NOT modify the original dataset.
# ============================================================


# ============================================================
# CELL 1 - Imports
# ============================================================

from pathlib import Path
from collections import Counter
import re
import unicodedata
import random

import numpy as np
import pandas as pd


# ============================================================
# CELL 2 - Paths
# ============================================================

# Adjust only if your folder is located elsewhere.
BASE_DIR = Path(r"D:\Degree\Hackathons\Amazon ML Challenge\student_resource")

DATA_DIR = BASE_DIR / "dataset"
TRAIN_DIR = DATA_DIR / "train"

M1_DIR = BASE_DIR / "m1_work"
REPORT_DIR = M1_DIR / "reports"
OUTPUT_DIR = M1_DIR / "outputs" / "normalization_analysis"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CELL 3 - Configuration
# ============================================================

# Deterministic seed so results are reproducible.
RANDOM_SEED = 42

# Maximum positive pairs to analyze.
#
# The full ground truth contains millions of positive pairs.
# We use a deterministic sample to keep this analysis practical.
MAX_POSITIVE_PAIRS = 500_000

# Number of negative pairs to generate.
#
# These are sampled from S1/S2 and S1/S3 pairs that are NOT
# present in ground truth.
MAX_NEGATIVE_PAIRS_PER_SOURCE = 500_000

# Number of examples saved for each transformation.
MAX_EXAMPLES_PER_CATEGORY = 100

# Minimum frequency before a normalized value is considered
# useful for collision analysis.
MIN_KEY_FREQUENCY = 2

rng = np.random.default_rng(RANDOM_SEED)

print("Base directory:")
print(BASE_DIR)

print("\nOutput directory:")
print(OUTPUT_DIR)


# ============================================================
# CELL 4 - Load datasets
# ============================================================

def load_tsv(path):
    print(f"Loading: {path}")
    df = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        na_filter=False
    )

    print(f"Rows: {len(df):,}")
    print(f"Columns: {list(df.columns)}")

    return df


s1 = load_tsv(TRAIN_DIR / "train_source1.tsv")
s2 = load_tsv(TRAIN_DIR / "train_source2.tsv")
s3 = load_tsv(TRAIN_DIR / "train_source3.tsv")
gt = load_tsv(TRAIN_DIR / "train_ground_truth.tsv")


# ============================================================
# CELL 5 - Basic validation
# ============================================================

required_s1 = {
    "entity_id",
    "business_name",
    "business_address",
    "country"
}

required_s2 = required_s1
required_s3 = required_s1

required_gt = {
    "source1_entity_id",
    "matched_entity_ids"
}

assert required_s1.issubset(s1.columns)
assert required_s2.issubset(s2.columns)
assert required_s3.issubset(s3.columns)
assert required_gt.issubset(gt.columns)

print("Dataset schema validation: PASSED")


# ============================================================
# CELL 6 - Build lookup dictionaries
# ============================================================

print("\nBuilding Source 2 lookup...")
s2_lookup = s2.set_index("entity_id").to_dict("index")

print("Building Source 3 lookup...")
s3_lookup = s3.set_index("entity_id").to_dict("index")

print("Lookup construction complete.")


# ============================================================
# CELL 7 - Parse ground truth
# ============================================================

def parse_matches(value):
    if value is None:
        return []

    value = str(value).strip()

    if not value:
        return []

    return [
        x.strip()
        for x in value.split(",")
        if x.strip()
    ]


gt_work = gt.copy()

gt_work["matched_ids"] = gt_work["matched_entity_ids"].map(parse_matches)

gt_work["match_count"] = gt_work["matched_ids"].map(len)

print("\nGround truth:")
print(f"Source 1 entities: {len(gt_work):,}")
print(
    f"Positive pairs: "
    f"{gt_work['match_count'].sum():,}"
)

print(
    "\nMatch-count distribution:"
)

print(
    gt_work["match_count"]
    .value_counts()
    .sort_index()
    .head(20)
)


# ============================================================
# CELL 8 - Expand positive pairs
# ============================================================

def expand_positive_pairs(gt_df, source_lookup, source_name):
    rows = []

    for row in gt_df.itertuples(index=False):
        s1_id = row.source1_entity_id

        for matched_id in row.matched_ids:

            record = source_lookup.get(matched_id)

            if record is None:
                continue

            rows.append({
                "source": source_name,
                "source1_entity_id": s1_id,
                "matched_entity_id": matched_id,

                "s1_name": s1_row_cache[s1_id]["business_name"],
                "s1_address": s1_row_cache[s1_id]["business_address"],
                "s1_country": s1_row_cache[s1_id]["country"],

                "matched_name": record["business_name"],
                "matched_address": record["business_address"],
                "matched_country": record["country"],
            })

    return pd.DataFrame(rows)


# Build S1 cache.
s1_row_cache = s1.set_index("entity_id").to_dict("index")

print("Expanding positive S2 pairs...")
positive_s2 = expand_positive_pairs(
    gt_work,
    s2_lookup,
    "S2"
)

print(f"S2 positive pairs: {len(positive_s2):,}")

print("\nExpanding positive S3 pairs...")
positive_s3 = expand_positive_pairs(
    gt_work,
    s3_lookup,
    "S3"
)

print(f"S3 positive pairs: {len(positive_s3):,}")


# ============================================================
# CELL 9 - Combine positives
# ============================================================

positive_pairs = pd.concat(
    [positive_s2, positive_s3],
    ignore_index=True
)

print(
    f"\nTotal positive pairs available: "
    f"{len(positive_pairs):,}"
)


# ============================================================
# CELL 10 - Deterministic positive sampling
# ============================================================

if len(positive_pairs) > MAX_POSITIVE_PAIRS:

    positive_pairs = (
        positive_pairs
        .sample(
            n=MAX_POSITIVE_PAIRS,
            random_state=RANDOM_SEED
        )
        .reset_index(drop=True)
    )

print(
    f"Positive pairs used for normalization analysis: "
    f"{len(positive_pairs):,}"
)

print(
    positive_pairs["source"]
    .value_counts()
)


# ============================================================
# CELL 11 - Normalization functions
# ============================================================

def safe_text(value):
    """
    Convert value to a safe string.
    """
    if value is None:
        return ""

    if isinstance(value, float) and np.isnan(value):
        return ""

    return str(value)


def unicode_normalize(value):
    """
    Unicode compatibility normalization.
    """
    value = safe_text(value)

    if not value:
        return ""

    return unicodedata.normalize("NFKC", value)


def casefold_text(value):
    """
    Unicode-aware lowercase transformation.
    """
    value = unicode_normalize(value)

    if not value:
        return ""

    return value.casefold()


def normalize_whitespace(value):
    """
    Collapse repeated whitespace.
    """
    value = casefold_text(value)

    if not value:
        return ""

    return re.sub(r"\s+", " ", value).strip()


def normalize_punctuation(value):
    """
    Replace punctuation with spaces.

    Important:
    We do NOT remove digits.
    """
    value = normalize_whitespace(value)

    if not value:
        return ""

    chars = []

    for char in value:

        category = unicodedata.category(char)

        if category.startswith("P"):
            chars.append(" ")
        else:
            chars.append(char)

    value = "".join(chars)

    return re.sub(r"\s+", " ", value).strip()


def alphanumeric_only(value):
    """
    Keep Unicode letters and digits.
    Remove punctuation and spaces.
    """
    value = unicode_normalize(value).casefold()

    if not value:
        return ""

    chars = []

    for char in value:

        if char.isalnum():
            chars.append(char)

    return "".join(chars)


def token_list(value):
    """
    Tokenize after punctuation normalization.
    """
    value = normalize_punctuation(value)

    if not value:
        return []

    return value.split()


def sorted_tokens(value):
    """
    Sort normalized tokens.

    Useful for diagnosing token-order variation.
    """
    tokens = token_list(value)

    if not tokens:
        return ""

    return " ".join(sorted(tokens))


def digit_signature(value):
    """
    Extract digits in order.

    This is diagnostic only.
    It should NOT be used as a standalone match key.
    """
    value = safe_text(value)

    return "".join(
        char
        for char in value
        if char.isdigit()
    )


def transformation_bundle(value):
    """
    Generate multiple representations.

    Original information is preserved.
    """
    return {
        "original": safe_text(value),
        "unicode": unicode_normalize(value),
        "casefold": casefold_text(value),
        "whitespace": normalize_whitespace(value),
        "punctuation": normalize_punctuation(value),
        "alphanumeric": alphanumeric_only(value),
        "sorted_tokens": sorted_tokens(value),
        "digits": digit_signature(value),
    }


# ============================================================
# CELL 12 - Test normalization functions
# ============================================================

test_values = [
    "ABC Pvt. Ltd.",
    "ABC  Pvt Ltd",
    "  ABC-PVT-LTD  ",
    "A/B/C",
    "Main Road, Pune - 411001",
    "Café Élite",
]

test_df = pd.DataFrame({
    "original": test_values
})

for transformation in [
    "unicode",
    "casefold",
    "whitespace",
    "punctuation",
    "alphanumeric",
    "sorted_tokens",
    "digits",
]:

    test_df[transformation] = test_df["original"].map(
        lambda x: transformation_bundle(x)[transformation]
    )

print(test_df.to_string(index=False))


# ============================================================
# CELL 13 - Positive transformation analysis
# ============================================================

TRANSFORMATIONS = [
    "original",
    "unicode",
    "casefold",
    "whitespace",
    "punctuation",
    "alphanumeric",
    "sorted_tokens",
    "digits",
]


def exact_match_rate(
    df,
    left_column,
    right_column,
    transform
):
    left = df[left_column].map(
        lambda x: transformation_bundle(x)[transform]
    )

    right = df[right_column].map(
        lambda x: transformation_bundle(x)[transform]
    )

    return float(
        (left == right).mean()
    )


positive_name_results = []

for transform in TRANSFORMATIONS:

    rate = exact_match_rate(
        positive_pairs,
        "s1_name",
        "matched_name",
        transform
    )

    positive_name_results.append({
        "transformation": transform,
        "exact_recovery_rate": rate,
        "recovered_pairs": int(
            round(rate * len(positive_pairs))
        )
    })


positive_name_results = pd.DataFrame(
    positive_name_results
)

positive_name_results["recovery_percent"] = (
    positive_name_results["exact_recovery_rate"] * 100
)


# Address

positive_address_results = []

for transform in TRANSFORMATIONS:

    rate = exact_match_rate(
        positive_pairs,
        "s1_address",
        "matched_address",
        transform
    )

    positive_address_results.append({
        "transformation": transform,
        "exact_recovery_rate": rate,
        "recovered_pairs": int(
            round(rate * len(positive_pairs))
        )
    })


positive_address_results = pd.DataFrame(
    positive_address_results
)

positive_address_results["recovery_percent"] = (
    positive_address_results["exact_recovery_rate"] * 100
)


print("\nNAME POSITIVE RECOVERY")
print(
    positive_name_results[
        ["transformation", "recovery_percent", "recovered_pairs"]
    ].to_string(index=False)
)

print("\nADDRESS POSITIVE RECOVERY")
print(
    positive_address_results[
        ["transformation", "recovery_percent", "recovered_pairs"]
    ].to_string(index=False)
)


# ============================================================
# CELL 14 - Save positive recovery
# ============================================================

positive_name_results.to_csv(
    OUTPUT_DIR / "positive_name_recovery.csv",
    index=False
)

positive_address_results.to_csv(
    OUTPUT_DIR / "positive_address_recovery.csv",
    index=False
)


# ============================================================
# CELL 15 - Calculate incremental recovery
# ============================================================

def add_incremental_recovery(df):
    df = df.copy()

    df["incremental_recovery_percent"] = (
        df["recovery_percent"]
        .diff()
        .fillna(df["recovery_percent"])
    )

    return df


positive_name_results = add_incremental_recovery(
    positive_name_results
)

positive_address_results = add_incremental_recovery(
    positive_address_results
)


print("\nIncremental NAME recovery:")
print(
    positive_name_results[
        [
            "transformation",
            "recovery_percent",
            "incremental_recovery_percent"
        ]
    ].to_string(index=False)
)

print("\nIncremental ADDRESS recovery:")
print(
    positive_address_results[
        [
            "transformation",
            "recovery_percent",
            "incremental_recovery_percent"
        ]
    ].to_string(index=False)
)


# ============================================================
# CELL 16 - Generate negative pairs
# ============================================================

def build_ground_truth_sets(gt_df):
    """
    Create:
        S1 -> matched S2 IDs
        S1 -> matched S3 IDs
    """

    s2_truth = {}
    s3_truth = {}

    for row in gt_df.itertuples(index=False):

        s1_id = row.source1_entity_id

        s2_truth[s1_id] = set()
        s3_truth[s1_id] = set()

        for entity_id in row.matched_ids:

            if entity_id.startswith("S2-"):
                s2_truth[s1_id].add(entity_id)

            elif entity_id.startswith("S3-"):
                s3_truth[s1_id].add(entity_id)

    return s2_truth, s3_truth


s2_truth, s3_truth = build_ground_truth_sets(gt_work)


# ============================================================
# CELL 17 - Negative-pair sampler
# ============================================================

def generate_negative_pairs(
    s1_df,
    target_df,
    truth_sets,
    source_name,
    n_pairs,
    seed=42
):

    local_rng = np.random.default_rng(seed)

    s1_ids = s1_df["entity_id"].to_numpy()
    target_ids = target_df["entity_id"].to_numpy()

    target_lookup = target_df.set_index("entity_id").to_dict("index")

    rows = []

    attempts = 0
    max_attempts = n_pairs * 20

    while len(rows) < n_pairs and attempts < max_attempts:

        batch_size = min(
            max(10_000, n_pairs - len(rows)),
            100_000
        )

        left_ids = local_rng.choice(
            s1_ids,
            size=batch_size,
            replace=True
        )

        right_ids = local_rng.choice(
            target_ids,
            size=batch_size,
            replace=True
        )

        for s1_id, target_id in zip(
            left_ids,
            right_ids
        ):

            attempts += 1

            if target_id in truth_sets.get(
                s1_id,
                set()
            ):
                continue

            s1_record = s1_row_cache[s1_id]
            target_record = target_lookup[target_id]

            rows.append({
                "source": source_name,
                "source1_entity_id": s1_id,
                "matched_entity_id": target_id,

                "s1_name": s1_record["business_name"],
                "s1_address": s1_record["business_address"],
                "s1_country": s1_record["country"],

                "matched_name": target_record["business_name"],
                "matched_address": target_record["business_address"],
                "matched_country": target_record["country"],
            })

            if len(rows) >= n_pairs:
                break

    negative_df = pd.DataFrame(rows)

    print(
        f"{source_name} negative pairs generated: "
        f"{len(negative_df):,}"
    )

    return negative_df


print("Generating negative S2 pairs...")

negative_s2 = generate_negative_pairs(
    s1,
    s2,
    s2_truth,
    "S2",
    MAX_NEGATIVE_PAIRS_PER_SOURCE,
    RANDOM_SEED
)

print("\nGenerating negative S3 pairs...")

negative_s3 = generate_negative_pairs(
    s1,
    s3,
    s3_truth,
    "S3",
    MAX_NEGATIVE_PAIRS_PER_SOURCE,
    RANDOM_SEED + 1
)


negative_pairs = pd.concat(
    [negative_s2, negative_s3],
    ignore_index=True
)

print(
    f"\nTotal negative pairs: "
    f"{len(negative_pairs):,}"
)


# ============================================================
# CELL 18 - Negative transformation equality
# ============================================================

def calculate_negative_collision_rates(
    negative_df,
    left_column,
    right_column
):

    rows = []

    for transform in TRANSFORMATIONS:

        left = negative_df[left_column].map(
            lambda x: transformation_bundle(x)[transform]
        )

        right = negative_df[right_column].map(
            lambda x: transformation_bundle(x)[transform]
        )

        equal = (
            (left == right)
            &
            (left != "")
        )

        collision_count = int(equal.sum())

        rows.append({
            "transformation": transform,
            "collision_pairs": collision_count,
            "collision_rate": (
                collision_count / len(negative_df)
                if len(negative_df)
                else 0
            )
        })

    return pd.DataFrame(rows)


negative_name_results = calculate_negative_collision_rates(
    negative_pairs,
    "s1_name",
    "matched_name"
)

negative_address_results = calculate_negative_collision_rates(
    negative_pairs,
    "s1_address",
    "matched_address"
)


negative_name_results["collision_percent"] = (
    negative_name_results["collision_rate"] * 100
)

negative_address_results["collision_percent"] = (
    negative_address_results["collision_rate"] * 100
)


print("\nNEGATIVE NAME COLLISIONS")
print(
    negative_name_results.to_string(index=False)
)

print("\nNEGATIVE ADDRESS COLLISIONS")
print(
    negative_address_results.to_string(index=False)
)


# ============================================================
# CELL 19 - Save collision results
# ============================================================

negative_name_results.to_csv(
    OUTPUT_DIR / "negative_name_collision.csv",
    index=False
)

negative_address_results.to_csv(
    OUTPUT_DIR / "negative_address_collision.csv",
    index=False
)


# ============================================================
# CELL 20 - Collision examples
# ============================================================

def collect_collision_examples(
    negative_df,
    left_column,
    right_column,
    max_examples=100
):

    all_examples = []

    for transform in TRANSFORMATIONS:

        if transform == "original":
            continue

        left_norm = negative_df[left_column].map(
            lambda x: transformation_bundle(x)[transform]
        )

        right_norm = negative_df[right_column].map(
            lambda x: transformation_bundle(x)[transform]
        )

        mask = (
            (left_norm == right_norm)
            &
            (left_norm != "")
        )

        subset = negative_df.loc[
            mask,
            [
                "source",
                "source1_entity_id",
                "matched_entity_id",
                left_column,
                right_column
            ]
        ].copy()

        if subset.empty:
            continue

        subset = subset.head(max_examples)

        subset.insert(
            0,
            "transformation",
            transform
        )

        subset["normalized_value"] = left_norm.loc[
            subset.index
        ].values

        all_examples.append(subset)

    if not all_examples:
        return pd.DataFrame()

    return pd.concat(
        all_examples,
        ignore_index=True
    )


name_collision_examples = collect_collision_examples(
    negative_pairs,
    "s1_name",
    "matched_name",
    MAX_EXAMPLES_PER_CATEGORY
)

address_collision_examples = collect_collision_examples(
    negative_pairs,
    "s1_address",
    "matched_address",
    MAX_EXAMPLES_PER_CATEGORY
)


name_collision_examples.to_csv(
    OUTPUT_DIR / "negative_name_collision.csv",
    index=False
)

address_collision_examples.to_csv(
    OUTPUT_DIR / "negative_address_collision.csv",
    index=False
)


# ============================================================
# CELL 21 - Country-conditioned analysis
# ============================================================

def country_conditioned_recovery(
    positive_df,
    left_column,
    right_column
):

    rows = []

    for country, group in positive_df.groupby(
        "s1_country",
        dropna=False
    ):

        for transform in TRANSFORMATIONS:

            left = group[left_column].map(
                lambda x: transformation_bundle(x)[transform]
            )

            right = group[right_column].map(
                lambda x: transformation_bundle(x)[transform]
            )

            rate = (
                (left == right).mean()
                if len(group)
                else 0
            )

            rows.append({
                "country": country,
                "transformation": transform,
                "pairs": len(group),
                "recovery_percent": rate * 100
            })

    return pd.DataFrame(rows)


country_name_results = country_conditioned_recovery(
    positive_pairs,
    "s1_name",
    "matched_name"
)

country_address_results = country_conditioned_recovery(
    positive_pairs,
    "s1_address",
    "matched_address"
)


print("\nCOUNTRY-CONDITIONED NAME RECOVERY")
print(
    country_name_results.to_string(index=False)
)

print("\nCOUNTRY-CONDITIONED ADDRESS RECOVERY")
print(
    country_address_results.to_string(index=False)
)


country_name_results.to_csv(
    OUTPUT_DIR / "country_name_recovery.csv",
    index=False
)

country_address_results.to_csv(
    OUTPUT_DIR / "country_address_recovery.csv",
    index=False
)


# ============================================================
# CELL 22 - Empty-value analysis
# ============================================================

def empty_value_analysis(
    positive_df,
    column
):

    total = len(positive_df)

    left_empty = (
        positive_df[column]
        .map(safe_text)
        .str.strip()
        .eq("")
    )

    return {
        "total_pairs": total,
        "s1_empty": int(left_empty.sum())
    }


empty_name = empty_value_analysis(
    positive_pairs,
    "s1_name"
)

empty_address = empty_value_analysis(
    positive_pairs,
    "s1_address"
)

print("\nEmpty S1 name:")
print(empty_name)

print("\nEmpty S1 address:")
print(empty_address)


# ============================================================
# CELL 23 - Identify transformations with recovery gain
# ============================================================

def transformation_summary(
    positive_results,
    negative_results
):

    summary = positive_results[
        [
            "transformation",
            "recovery_percent",
            "incremental_recovery_percent"
        ]
    ].copy()

    collision_lookup = (
        negative_results
        .set_index("transformation")
        ["collision_percent"]
        .to_dict()
    )

    summary["negative_collision_percent"] = (
        summary["transformation"]
        .map(collision_lookup)
        .fillna(0)
    )

    return summary


name_summary = transformation_summary(
    positive_name_results,
    negative_name_results
)

address_summary = transformation_summary(
    positive_address_results,
    negative_address_results
)


print("\nNAME TRANSFORMATION SUMMARY")
print(
    name_summary.to_string(index=False)
)

print("\nADDRESS TRANSFORMATION SUMMARY")
print(
    address_summary.to_string(index=False)
)


# ============================================================
# CELL 24 - Representative positive examples
# ============================================================

def collect_positive_examples(
    positive_df,
    left_column,
    right_column,
    max_examples=100
):

    examples = []

    for transform in TRANSFORMATIONS:

        if transform == "original":
            continue

        left_norm = positive_df[left_column].map(
            lambda x: transformation_bundle(x)[transform]
        )

        right_norm = positive_df[right_column].map(
            lambda x: transformation_bundle(x)[transform]
        )

        # Cases where normalization makes them equal,
        # but raw strings were different.
        mask = (
            (left_norm == right_norm)
            &
            (
                positive_df[left_column].map(safe_text)
                !=
                positive_df[right_column].map(safe_text)
            )
            &
            (left_norm != "")
        )

        subset = positive_df.loc[
            mask,
            [
                "source",
                "source1_entity_id",
                "matched_entity_id",
                left_column,
                right_column
            ]
        ].copy()

        if subset.empty:
            continue

        subset = subset.head(max_examples)

        subset.insert(
            0,
            "transformation",
            transform
        )

        subset["normalized_value"] = left_norm.loc[
            subset.index
        ].values

        examples.append(subset)

    if not examples:
        return pd.DataFrame()

    return pd.concat(
        examples,
        ignore_index=True
    )


name_positive_examples = collect_positive_examples(
    positive_pairs,
    "s1_name",
    "matched_name",
    MAX_EXAMPLES_PER_CATEGORY
)

address_positive_examples = collect_positive_examples(
    positive_pairs,
    "s1_address",
    "matched_address",
    MAX_EXAMPLES_PER_CATEGORY
)


positive_examples = pd.concat(
    [
        name_positive_examples.assign(field="business_name"),
        address_positive_examples.assign(field="business_address")
    ],
    ignore_index=True
)


positive_examples.to_csv(
    OUTPUT_DIR / "transformation_examples.csv",
    index=False
)


# ============================================================
# CELL 25 - Compare positive recovery vs negative collision
# ============================================================

name_comparison = name_summary.copy()

name_comparison["field"] = "business_name"

address_comparison = address_summary.copy()

address_comparison["field"] = "business_address"

transformation_comparison = pd.concat(
    [
        name_comparison,
        address_comparison
    ],
    ignore_index=True
)

print(
    "\nTRANSFORMATION RECOVERY / COLLISION COMPARISON"
)

print(
    transformation_comparison[
        [
            "field",
            "transformation",
            "recovery_percent",
            "incremental_recovery_percent",
            "negative_collision_percent"
        ]
    ].to_string(index=False)
)


transformation_comparison.to_csv(
    OUTPUT_DIR / "transformation_comparison.csv",
    index=False
)


# ============================================================
# CELL 26 - Generate recommendations
# ============================================================

def generate_recommendation_rows(
    summary_df,
    field_name
):

    rows = []

    for row in summary_df.itertuples(index=False):

        transform = row.transformation
        recovery = row.recovery_percent
        incremental = row.incremental_recovery_percent
        collision = row.negative_collision_percent

        if transform == "original":

            decision = "BASELINE"

        elif collision == 0 and incremental > 0:

            decision = "LOW_OBSERVED_COLLISION"

        elif collision < 0.01 and incremental > 0:

            decision = "REVIEW"

        elif collision < 0.1 and incremental > 0:

            decision = "CAREFUL_REVIEW"

        else:

            decision = "HIGH_COLLISION_REVIEW"

        rows.append({
            "field": field_name,
            "transformation": transform,
            "positive_recovery_percent": recovery,
            "incremental_recovery_percent": incremental,
            "negative_collision_percent": collision,
            "initial_decision": decision
        })

    return pd.DataFrame(rows)


recommendation_name = generate_recommendation_rows(
    name_summary,
    "business_name"
)

recommendation_address = generate_recommendation_rows(
    address_summary,
    "business_address"
)

recommendations = pd.concat(
    [
        recommendation_name,
        recommendation_address
    ],
    ignore_index=True
)


recommendations.to_csv(
    OUTPUT_DIR / "normalization_recommendations.csv",
    index=False
)


# ============================================================
# CELL 27 - Markdown report generation
# ============================================================

def fmt_pct(value):
    return f"{value:.2f}%"


report_lines = []

report_lines.append("# Normalization Analysis Report")
report_lines.append("")
report_lines.append(
    "Amazon ML Challenge 2026 — Business Entity Resolution"
)
report_lines.append("")

report_lines.append("## 1. Purpose")
report_lines.append("")
report_lines.append(
    "This analysis evaluates candidate normalization "
    "transformations using sampled positive matched pairs "
    "and sampled negative non-matching pairs."
)
report_lines.append("")
report_lines.append(
    "The objective is to identify transformations that recover "
    "true variations while measuring the observed collision risk "
    "on non-matching pairs."
)
report_lines.append("")

report_lines.append("## 2. Dataset used")
report_lines.append("")
report_lines.append(
    f"- Positive pairs analyzed: {len(positive_pairs):,}"
)
report_lines.append(
    f"- Negative pairs analyzed: {len(negative_pairs):,}"
)
report_lines.append(
    f"- Random seed: {RANDOM_SEED}"
)
report_lines.append("")

report_lines.append("## 3. Candidate transformations")
report_lines.append("")
report_lines.append(
    "The following representations were evaluated:"
)
report_lines.append("")
for transform in TRANSFORMATIONS:
    report_lines.append(f"- `{transform}`")
report_lines.append("")

report_lines.append("## 4. Business-name recovery")
report_lines.append("")
report_lines.append(
    "| Transformation | Recovery | Incremental Recovery | "
    "Negative Collision |"
)
report_lines.append(
    "|---|---:|---:|---:|"
)

for row in name_summary.itertuples(index=False):

    report_lines.append(
        f"| `{row.transformation}` | "
        f"{fmt_pct(row.recovery_percent)} | "
        f"{fmt_pct(row.incremental_recovery_percent)} | "
        f"{fmt_pct(row.negative_collision_percent)} |"
    )

report_lines.append("")

report_lines.append("## 5. Business-address recovery")
report_lines.append("")
report_lines.append(
    "| Transformation | Recovery | Incremental Recovery | "
    "Negative Collision |"
)
report_lines.append(
    "|---|---:|---:|---:|"
)

for row in address_summary.itertuples(index=False):

    report_lines.append(
        f"| `{row.transformation}` | "
        f"{fmt_pct(row.recovery_percent)} | "
        f"{fmt_pct(row.incremental_recovery_percent)} | "
        f"{fmt_pct(row.negative_collision_percent)} |"
    )

report_lines.append("")

report_lines.append("## 6. Interpretation")
report_lines.append("")

report_lines.append(
    "The recovery percentage measures how many sampled positive "
    "pairs become exactly equal after a transformation."
)

report_lines.append("")

report_lines.append(
    "The negative collision percentage measures how many sampled "
    "non-matching pairs become exactly equal after a transformation."
)

report_lines.append("")

report_lines.append(
    "A transformation should not be selected only because it "
    "increases positive recovery. Collision behavior must also "
    "be considered because false merges are costly for this task."
)

report_lines.append("")

report_lines.append("## 7. Important implementation principle")
report_lines.append("")

report_lines.append(
    "Normalization should generate multiple representations "
    "rather than replacing the original value."
)

report_lines.append("")

report_lines.append(
    "Recommended conceptual structure:"
)

report_lines.append("")
report_lines.append(
    "Original → Unicode → Casefold → Punctuation → "
    "Alphanumeric → Token → Sorted Tokens → Digits"
)

report_lines.append("")

report_lines.append(
    "The original representation should remain available for "
    "final matching and explainability."
)

report_lines.append("")

report_lines.append("## 8. Negative-pair analysis limitations")
report_lines.append("")

report_lines.append(
    "Negative pairs were sampled rather than exhaustively "
    "enumerated because the complete cross-source Cartesian "
    "product is extremely large."
)

report_lines.append("")

report_lines.append(
    "Therefore, observed collision rates describe the sampled "
    "negative population and should not be interpreted as an "
    "exact full-dataset collision probability."
)

report_lines.append("")

report_lines.append("## 9. Files generated")
report_lines.append("")

for filename in [
    "positive_name_recovery.csv",
    "positive_address_recovery.csv",
    "negative_name_collision.csv",
    "negative_address_collision.csv",
    "country_name_recovery.csv",
    "country_address_recovery.csv",
    "transformation_examples.csv",
    "transformation_comparison.csv",
    "normalization_recommendations.csv",
]:

    report_lines.append(
        f"- `{filename}`"
    )

report_lines.append("")

report_lines.append("## 10. Next stage")
report_lines.append("")

report_lines.append(
    "Use the measured recovery and collision results to design "
    "the production `text_normalizer.py` module."
)

report_lines.append("")

report_lines.append(
    "The production implementation should preserve multiple "
    "representations and avoid aggressive transformations that "
    "can collapse distinct businesses."
)

report_lines.append("")


report_path = REPORT_DIR / "NORMALIZATION_ANALYSIS.md"

report_path.write_text(
    "\n".join(report_lines),
    encoding="utf-8"
)

print("\nReport written to:")
print(report_path)


# ============================================================
# CELL 28 - Final summary
# ============================================================

print("\n" + "=" * 70)
print("TASK 4 COMPLETED")
print("=" * 70)

print(
    f"\nPositive pairs analyzed : {len(positive_pairs):,}"
)

print(
    f"Negative pairs analyzed : {len(negative_pairs):,}"
)

print(
    "\nReports:"
)

print(
    REPORT_DIR / "NORMALIZATION_ANALYSIS.md"
)

print(
    "\nCSV outputs:"
)

for file in sorted(OUTPUT_DIR.glob("*.csv")):
    print(" -", file.name)

print("\nNext step:")
print(
    "Review normalization_recommendations.csv and "
    "NORMALIZATION_ANALYSIS.md before implementing "
    "text_normalizer.py."
)

Base directory:
D:\Degree\Hackathons\Amazon ML Challenge\student_resource

Output directory:
D:\Degree\Hackathons\Amazon ML Challenge\student_resource\m1_work\outputs\normalization_analysis
Loading: D:\Degree\Hackathons\Amazon ML Challenge\student_resource\dataset\train\train_source1.tsv
Rows: 2,206,821
Columns: ['entity_id', 'business_name', 'business_address', 'country']
Loading: D:\Degree\Hackathons\Amazon ML Challenge\student_resource\dataset\train\train_source2.tsv
Rows: 5,034,616
Columns: ['entity_id', 'business_name', 'business_address', 'country']
Loading: D:\Degree\Hackathons\Amazon ML Challenge\student_resource\dataset\train\train_source3.tsv
Rows: 5,285,603
Columns: ['entity_id', 'business_name', 'business_address', 'country']
Loading: D:\Degree\Hackathons\Amazon ML Challenge\student_resource\dataset\train\train_ground_truth.tsv
Rows: 2,206,821
Columns: ['source1_entity_id', 'matched_entity_ids']
Dataset schema validation: PASSED

Building Source 2 lookup...
Building Source 